# Phase 4 — OpenTargets Platform ingestion

This notebook walks through the full OpenTargets ingestion pipeline that
populates `data/kg/nodes/` and `data/kg/edges/` Parquet files from the
OpenTargets Platform bulk data.

## What gets ingested

| OT Dataset | Output |
|---|---|
| `target` | `nodes/gene.parquet` (Ensembl IDs + xrefs) |
| `disease` | `nodes/disease.parquet` (EFO IDs + xrefs) |
| `molecule` / `drug` | `nodes/molecule.parquet` (ChEMBL IDs + xrefs) |
| `interaction` | `edges/protein_interacts_protein.parquet` |
| `evidence` | `edges/disease_associated_gene.parquet`, `molecule_treats_disease.parquet`, … |
| `go` | `nodes/pathway.parquet` (GO terms) + `edges/pathway_contains_gene.parquet` |
| `reactome` | `nodes/pathway.parquet` (Reactome) + `edges/pathway_child_of_pathway.parquet` |
| `literature` | `nodes/paper.parquet` + `edges/paper_mentions_*.parquet` |
| `indication` | `edges/molecule_treats_disease.parquet`, `molecule_contraindicates_disease.parquet` |
| `mechanismOfAction` | `edges/molecule_targets_protein.parquet` |

## Credibility scoring

| Score | Meaning | OT datatypes |
|---|---|---|
| 3 | Established fact | `known_drug`, `chembl`, `eva`, `clingen` |
| 2 | Multiple independent evidence | `genetic_association`, `somatic_mutation`, `l2g`, `crispr_screen` |
| 1 | Single evidence | `europepmc`, `phenodigm`, `literature` |

## Prerequisites

```bash
# Phase 3 migration must have run first:
uv run python -m manage_db.kg_migrate --data-dir ./data

# Then run OT ingestion:
uv run python -m manage_db.ingest_opentargets --data-dir ./data --release latest
```

> **Note**: The full ingestion downloads ~20–50 GB of Parquet data from
> EBI FTP (depending on release). It is resumable — re-running skips already
> downloaded files.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import pyarrow.parquet as pq

# Import ingestion helpers
from manage_db.ingest_opentargets import (
    ingest_targets,
    ingest_diseases,
    ingest_drugs,
    ingest_interactions,
    ingest_evidence,
    ingest_go,
    ingest_reactome,
    ingest_literature,
    ingest_indication,
    ingest_mechanism_of_action,
    run,
    ALL_DATASETS,
    DATATYPE_CREDIBILITY,
)
from txdata_download import get_latest_opentargets_release

DATA_DIR = ROOT / "data"
OT_DIR   = DATA_DIR / "opentargets"
KG_DIR   = DATA_DIR / "kg"

print("ROOT    :", ROOT)
print("OT_DIR  :", OT_DIR)
print("KG_DIR  :", KG_DIR)

## 0. Discover the latest OpenTargets release

In [ ]:
from txdata_download import get_opentargets_releases, list_opentargets_datasets

releases = get_opentargets_releases()
latest   = releases[-1]

print(f"Available releases: {releases[-5:]} … (showing last 5)")
print(f"Latest            : {latest}")

datasets = list_opentargets_datasets(latest)
print(f"\nDatasets in {latest} ({len(datasets)} total):")
for ds in datasets:
    print(f"  {ds}")

## 1. Download OpenTargets datasets

Downloads are resumable — running this cell again skips already-complete
datasets (`.ot_complete` marker present).

In [ ]:
# Check what is already downloaded.
OT_DIR.mkdir(parents=True, exist_ok=True)

status = []
for ds in ALL_DATASETS:
    ds_dir  = OT_DIR / ds
    exists  = ds_dir.exists()
    marker  = (ds_dir / ".ot_complete").exists() if exists else False
    n_files = len(list(ds_dir.glob("*.parquet"))) if exists else 0
    status.append({"dataset": ds, "dir_exists": exists, "complete": marker, "parquet_files": n_files})

pd.DataFrame(status)

In [ ]:
# Download all required datasets (skips complete ones automatically).
# Set RELEASE to a specific tag like "25.06" to pin the version.
RELEASE = "latest"

run(
    data_dir=DATA_DIR,
    datasets=ALL_DATASETS,
    release=RELEASE,
    download=True,
    workers=8,
)

## 2. Inspect raw OT data

Before ingestion, let's peek at the raw Parquet schemas.

In [ ]:
# --- target dataset ---
target_files = sorted((OT_DIR / "target").glob("*.parquet"))
print(f"target: {len(target_files)} parquet files")

tgt_sample = pd.read_parquet(target_files[0], columns=[
    "id", "approvedSymbol", "approvedName", "biotype",
    "dbXrefs", "proteinIds", "transcriptIds",
])
print(f"Rows in first file: {len(tgt_sample):,}")
tgt_sample.head(3)

In [ ]:
# --- disease dataset ---
dis_files = sorted((OT_DIR / "disease").glob("*.parquet"))
print(f"disease: {len(dis_files)} parquet files")

dis_sample = pd.read_parquet(dis_files[0], columns=[
    "id", "name", "dbXRefs", "synonyms", "therapeuticAreas",
])
print(f"Rows in first file: {len(dis_sample):,}")
dis_sample.head(3)

In [ ]:
# --- molecule / drug dataset ---
mol_dir   = OT_DIR / "molecule"
drug_dir  = OT_DIR / "drug"
mol_files = sorted((mol_dir if mol_dir.exists() else drug_dir).glob("*.parquet"))
print(f"molecule: {len(mol_files)} parquet files")

mol_sample = pd.read_parquet(mol_files[0])
print(f"Columns: {list(mol_sample.columns[:10])} …")
mol_sample[["id", "name", "drugType", "isApproved"]].head(3)

In [ ]:
# --- evidence: sample datatypeId distribution ---
ev_files = sorted((OT_DIR / "evidence").glob("*.parquet"))
print(f"evidence: {len(ev_files)} parquet files")

# Read just datatypeId from a few files to get the distribution.
sample_dfs = []
for f in ev_files[:3]:
    sample_dfs.append(pd.read_parquet(f, columns=["datatypeId", "score"]))

ev_sample = pd.concat(sample_dfs)
print("\ndatatypeId → credibility mapping:")
dt_counts = ev_sample["datatypeId"].value_counts().rename("count")
dt_cred   = pd.Series(DATATYPE_CREDIBILITY, name="credibility")
pd.concat([dt_counts, dt_cred], axis=1).dropna(subset=["count"]).sort_values("count", ascending=False)

## 3. Ingest node datasets

Each `ingest_*` function reads the raw OT Parquet files and writes/merges
into `data/kg/nodes/{node_type}.parquet`.

In [ ]:
n_targets = ingest_targets(OT_DIR, KG_DIR)
print(f"Ingested {n_targets:,} gene nodes")

In [ ]:
gene_df = pd.read_parquet(KG_DIR / "nodes" / "gene.parquet")
print(f"Total gene nodes: {len(gene_df):,}")
print(f"Columns: {list(gene_df.columns)}")
gene_df.head(3)

In [ ]:
n_diseases = ingest_diseases(OT_DIR, KG_DIR)
print(f"Ingested {n_diseases:,} disease nodes")

In [ ]:
disease_df = pd.read_parquet(KG_DIR / "nodes" / "disease.parquet")
print(f"Total disease nodes: {len(disease_df):,}")
print(f"Columns: {list(disease_df.columns)}")

# Breakdown of ID prefixes
disease_df["id_prefix"] = disease_df["id"].str.split("_").str[0]
disease_df["id_prefix"].value_counts().head()

In [ ]:
n_drugs = ingest_drugs(OT_DIR, KG_DIR)
print(f"Ingested {n_drugs:,} molecule nodes")

In [ ]:
mol_df = pd.read_parquet(KG_DIR / "nodes" / "molecule.parquet")
print(f"Total molecule nodes: {len(mol_df):,}")
print(f"Columns: {list(mol_df.columns)}")

# How many approved drugs?
if "isApproved" in mol_df.columns:
    print(f"Approved drugs: {mol_df['isApproved'].sum():,}")

mol_df.head(3)

## 4. Ingest edge datasets

In [ ]:
n_ppi = ingest_interactions(OT_DIR, KG_DIR)
print(f"Ingested {n_ppi:,} protein–protein interaction edges")

In [ ]:
ppi_df = pd.read_parquet(KG_DIR / "edges" / "protein_interacts_protein.parquet")
print(f"Total PPI edges: {len(ppi_df):,}")
print(f"Columns: {list(ppi_df.columns)}")
ppi_df[["x_id", "y_id", "credibility", "source"]].head(3)

In [ ]:
ev_counts = ingest_evidence(OT_DIR, KG_DIR)
print("Evidence edges written:")
for rel, cnt in sorted(ev_counts.items()):
    print(f"  {rel:<50}  {cnt:>9,}")

In [ ]:
n_go_nodes, n_go_edges = ingest_go(OT_DIR, KG_DIR)
print(f"Ingested {n_go_nodes:,} GO pathway nodes, {n_go_edges:,} pathway_contains_gene edges")

In [ ]:
n_rea_nodes, n_rea_edges = ingest_reactome(OT_DIR, KG_DIR)
print(f"Ingested {n_rea_nodes:,} Reactome pathway nodes, {n_rea_edges:,} pathway_child_of_pathway edges")

In [ ]:
pw_df = pd.read_parquet(KG_DIR / "nodes" / "pathway.parquet")
print(f"Total pathway nodes: {len(pw_df):,}")

# Source breakdown (GO vs Reactome)
print("\nSource breakdown:")
print(pw_df["source"].value_counts().to_string())

# ID prefix breakdown
pw_df["id_prefix"] = pw_df["id"].str.split(":").str[0]
print("\nID prefix breakdown:")
print(pw_df["id_prefix"].value_counts().to_string())

In [ ]:
n_ind, n_ci = ingest_indication(OT_DIR, KG_DIR)
print(f"Ingested {n_ind:,} molecule_treats_disease, {n_ci:,} molecule_contraindicates_disease edges")

In [ ]:
n_moa = ingest_mechanism_of_action(OT_DIR, KG_DIR)
print(f"Ingested {n_moa:,} molecule_targets_protein edges")

In [ ]:
# Literature is the largest dataset — this may take several minutes.
n_papers, n_mentions = ingest_literature(OT_DIR, KG_DIR)
print(f"Ingested {n_papers:,} paper nodes, {n_mentions:,} mention edges")

## 5. Post-ingestion summary

Inspect the final state of the `data/kg/` Parquet store.

In [ ]:
node_dir = KG_DIR / "nodes"
node_rows = []
for p in sorted(node_dir.glob("*.parquet")):
    meta = pq.read_metadata(p)
    node_rows.append({
        "node_type": p.stem,
        "rows": meta.num_rows,
        "size_MB": round(p.stat().st_size / 1e6, 1),
    })

node_summary = pd.DataFrame(node_rows).sort_values("rows", ascending=False)
print(f"Total node Parquet files: {len(node_summary)}")
print(f"Total nodes: {node_summary['rows'].sum():,}")
node_summary

In [ ]:
edge_dir = KG_DIR / "edges"
edge_rows = []
for p in sorted(edge_dir.glob("*.parquet")):
    meta = pq.read_metadata(p)
    edge_rows.append({
        "relation": p.stem,
        "rows": meta.num_rows,
        "size_MB": round(p.stat().st_size / 1e6, 1),
    })

edge_summary = pd.DataFrame(edge_rows).sort_values("rows", ascending=False)
print(f"Total edge Parquet files: {len(edge_summary)}")
print(f"Total edges: {edge_summary['rows'].sum():,}")
edge_summary

In [ ]:
# Credibility distribution across all edge files.
cred_counts = {}
for p in sorted(edge_dir.glob("*.parquet")):
    df = pd.read_parquet(p, columns=["credibility"])
    for cred, cnt in df["credibility"].value_counts().items():
        cred_counts[cred] = cred_counts.get(cred, 0) + cnt

total_edges = sum(cred_counts.values())
print("Credibility distribution across all edges:")
for cred in sorted(cred_counts):
    pct = 100 * cred_counts[cred] / total_edges
    label = {1: "single evidence", 2: "multi-evidence", 3: "established fact"}[cred]
    print(f"  {cred} ({label:<22}): {cred_counts[cred]:>10,}  ({pct:.1f}%)")

In [ ]:
# Source breakdown across all edge files.
source_counts = {}
for p in sorted(edge_dir.glob("*.parquet")):
    df = pd.read_parquet(p, columns=["source"])
    for src, cnt in df["source"].value_counts().items():
        source_counts[src] = source_counts.get(src, 0) + cnt

pd.Series(source_counts, name="count").sort_values(ascending=False).to_frame()

## 6. Spot-checks

Validate a few specific known relationships.

In [ ]:
# Imatinib (CHEMBL941) should appear as a molecule targeting ABL1 (ENSG00000097007).
moa_df = pd.read_parquet(KG_DIR / "edges" / "molecule_targets_protein.parquet")
imatinib_targets = moa_df[moa_df["x_id"] == "CHEMBL941"]
print(f"Imatinib (CHEMBL941) targets: {len(imatinib_targets)}")
imatinib_targets[["x_id", "y_id", "relation", "credibility"]].head(10)

In [ ]:
# BRCA1 (ENSG00000012048) should have disease associations.
dag_df = pd.read_parquet(KG_DIR / "edges" / "disease_associated_gene.parquet")
brca1_assoc = dag_df[dag_df["y_id"] == "ENSG00000012048"]
print(f"Diseases associated with BRCA1: {len(brca1_assoc):,}")
print("\nCredibility breakdown:")
print(brca1_assoc["credibility"].value_counts().to_string())
brca1_assoc[["x_id", "y_id", "credibility", "source"]].head(5)

In [ ]:
# Check disease nodes for breast cancer (EFO_0000305 / MONDO equivalents).
disease_df = pd.read_parquet(KG_DIR / "nodes" / "disease.parquet")
bc = disease_df[disease_df["name"].str.contains("breast cancer", case=False, na=False)]
print(f"'breast cancer' disease nodes: {len(bc)}")
bc[["id", "name", "source"]].head()

In [ ]:
# Dangling edge check: sample 1000 edges and verify both endpoint IDs exist.
import random

# Build set of all known node IDs.
all_node_ids: set[str] = set()
for p in (KG_DIR / "nodes").glob("*.parquet"):
    df = pd.read_parquet(p, columns=["id"])
    all_node_ids.update(df["id"].tolist())

print(f"Total distinct node IDs: {len(all_node_ids):,}")

# Sample edges.
dangling = []
edge_files = list((KG_DIR / "edges").glob("*.parquet"))
sample_file = random.choice(edge_files)
edge_sample = pd.read_parquet(sample_file, columns=["x_id", "y_id"]).sample(
    min(1000, len(pd.read_parquet(sample_file))), random_state=42
)

missing_x = ~edge_sample["x_id"].isin(all_node_ids)
missing_y = ~edge_sample["y_id"].isin(all_node_ids)

print(f"\nSampled {len(edge_sample)} edges from {sample_file.name}:")
print(f"  Missing x_id: {missing_x.sum()}")
print(f"  Missing y_id: {missing_y.sum()}")

if missing_x.sum() > 0 or missing_y.sum() > 0:
    print("\nDangling edges (sample):")
    dangling_df = edge_sample[missing_x | missing_y]
    print(dangling_df.head(10).to_string())
else:
    print("  → No dangling edges in sample. ✓")

## 7. One-shot CLI equivalent

The full ingestion above can be reproduced from the command line:

```bash
# Download + ingest all datasets from the latest release:
uv run python -m manage_db.ingest_opentargets \
    --data-dir ./data \
    --release latest \
    --workers 8

# Ingest only specific datasets (skip download):
uv run python -m manage_db.ingest_opentargets \
    --data-dir ./data \
    --no-download \
    --datasets target disease molecule evidence interaction

# Pin a specific release:
uv run python -m manage_db.ingest_opentargets \
    --data-dir ./data \
    --release 25.06
```

After ingestion, the full KG is available in `data/kg/` and ready for
`KGLoader` (Phase 8).